In [ ]:
# Bài 18 — Domain Shift: huấn luyện trên IMDb, kiểm tra trên IMDb và Yelp
%pip -q install datasets transformers scikit-learn

import random, time, numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# IMDb là miền huấn luyện; Yelp polarity là miền đánh giá ngoài miền.
imdb = load_dataset("stanfordnlp/imdb")
yelp = load_dataset("fancyzhx/yelp_polarity")

def balanced_sample(ds, n, seed, offset=0):
    """Lấy số mẫu cân bằng giữa hai nhãn, có thể tái lập."""
    n_per_class = n // 2
    chosen = []
    labels = np.asarray(ds["label"])
    rng = np.random.default_rng(seed)
    for label in (0, 1):
        idx = np.flatnonzero(labels == label)
        rng.shuffle(idx)
        chosen.extend(idx[offset:offset+n_per_class].tolist())
    rng.shuffle(chosen)
    return ds.select(chosen)

train_ds = balanced_sample(imdb["train"], 1000, SEED)
# Validation và test IMDb được lấy từ các phần khác nhau của official test split.
val_ds = balanced_sample(imdb["test"], 500, SEED + 1, offset=0)
in_ds = balanced_sample(imdb["test"], 500, SEED + 2, offset=250)
out_ds = balanced_sample(yelp["test"], 500, SEED + 3)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LENGTH = 128

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

cols = ["text", "label"]
train_ds = train_ds.map(tokenize, batched=True, remove_columns=[c for c in train_ds.column_names if c not in ("label",)])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=[c for c in val_ds.column_names if c not in ("label",)])
in_ds = in_ds.map(tokenize, batched=True, remove_columns=[c for c in in_ds.column_names if c not in ("label",)])
out_ds = out_ds.map(tokenize, batched=True, remove_columns=[c for c in out_ds.column_names if c not in ("label",)])
for dataset in (train_ds, val_ds, in_ds, out_ds):
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collator)

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collator)
    model.eval(); y_true=[]; y_pred=[]
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").cpu().numpy()
            batch = {k:v.to(device) for k,v in batch.items()}
            preds = model(**batch).logits.argmax(dim=-1).cpu().numpy()
            y_true.extend(labels.tolist()); y_pred.extend(preds.tolist())
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r,
            "f1": f1, "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()}

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
start = time.time(); model.train()
for step, batch in enumerate(train_loader, start=1):
    batch = {k:v.to(device) for k,v in batch.items()}
    loss = model(**batch).loss
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if step % 10 == 0 or step == len(train_loader):
        print(f"step {step}/{len(train_loader)} | loss={loss.item():.4f}")
train_seconds = time.time() - start
val_metrics = evaluate(model, val_ds)
in_metrics = evaluate(model, in_ds)
out_metrics = evaluate(model, out_ds)

print("\n=== Domain shift results (balanced 500-example test sets) ===")
print(f"Train: IMDb 1,000 balanced reviews | max_length={MAX_LENGTH} | epochs=1 | seed={SEED}")
print(f"IMDb in-domain:  Accuracy={in_metrics['accuracy']:.4f} Precision={in_metrics['precision']:.4f} Recall={in_metrics['recall']:.4f} F1={in_metrics['f1']:.4f}")
print("IMDb confusion matrix [true 0/1 × pred 0/1]:", in_metrics["confusion_matrix"])
print(f"Yelp out-domain:  Accuracy={out_metrics['accuracy']:.4f} Precision={out_metrics['precision']:.4f} Recall={out_metrics['recall']:.4f} F1={out_metrics['f1']:.4f}")
print("Yelp confusion matrix [true 0/1 × pred 0/1]:", out_metrics["confusion_matrix"])
print(f"Validation IMDb:  Accuracy={val_metrics['accuracy']:.4f} F1={val_metrics['f1']:.4f}")
print(f"Training time: {train_seconds:.1f}s")
print("\nKết luận dựa trên số liệu: so sánh IMDb in-domain với Yelp out-domain để xem khả năng giảm chất lượng khi chuyển miền.")


Device: cpu | GPU: CPU


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (21 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


step 10/63 | loss=0.7274
step 20/63 | loss=0.6219
step 30/63 | loss=0.5937
step 40/63 | loss=0.6401
step 50/63 | loss=0.4845
step 60/63 | loss=0.3430
step 63/63 | loss=0.5097

=== Domain shift results (balanced 500-example test sets) ===
Train: IMDb 1,000 balanced reviews | max_length=128 | epochs=1 | seed=42
IMDb in-domain:  Accuracy=0.8280 Precision=0.8254 Recall=0.8320 F1=0.8287
IMDb confusion matrix [true 0/1 × pred 0/1]: [[206, 44], [42, 208]]
Yelp out-domain:  Accuracy=0.8560 Precision=0.8069 Recall=0.9360 F1=0.8667
Yelp confusion matrix [true 0/1 × pred 0/1]: [[194, 56], [16, 234]]
Validation IMDb:  Accuracy=0.8320 F1=0.8346
Training time: 875.3s

Kết luận dựa trên số liệu: so sánh IMDb in-domain với Yelp out-domain để xem khả năng giảm chất lượng khi chuyển miền.
